##Configuração do Ambiente

Cria a estrutura do projeto e importa as bibliotecas necessárias.

In [ ]:
# Bibliotecas
import pandas as pd
import os
import zipfile
import urllib.request
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import numpy as np

##1. Carregamento dos Dados

- Ler o arquivo CSV
- Verificar se foi carregado corretamente
- Visualizar as primeiras linhas do dataset

In [ ]:
def carregar_dataset(caminho):
    encodings = ['utf-8', 'latin-1', 'cp1252']

    for enc in encodings:
        try:
            df = pd.read_csv(caminho, encoding=enc)
            print(f"Carregado com encoding: {enc}")
            return df
        except:
            print(f"Falhou com {enc}")

    return None

df = carregar_dataset("train.csv")

Falhou com utf-8
Falhou com latin-1
Falhou com cp1252


###1.1 Verificação Inicial dos Dados

- Tipos de dados
- Valores nulos
- Estrutura geral do dataset

In [ ]:
# Informações gerais
df.info()

# Estatísticas
df.describe()

# Valores nulos
df.isnull().sum()

AttributeError: 'NoneType' object has no attribute 'info'

##2. Análise Exploratória dos Dados

- Estrutura do dataset
- Tipos de dados
- Distribuição das vendas

In [ ]:
df.info()
df.describe()

##3. Criação dos Atributos

- Tempo (dia, mês, dia da semana)
- Lags (valores passados)
- Médias móveis

In [ ]:
def criar_features(df):
    df = df.copy()

    df['date'] = pd.to_datetime(df['date'])

    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['dayofweek'] = df['date'].dt.dayofweek
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

    df = df.sort_values(['store','item','date'])

    df['lag_1'] = df.groupby(['store','item'])['sales'].shift(1)
    df['lag_7'] = df.groupby(['store','item'])['sales'].shift(7)

    df['rolling_mean_7'] = df.groupby(['store','item'])['sales'].shift(1).rolling(7).mean()

    df = df.dropna()

    return df

##4. Divisão dos Dados

Divir os dados respeitando o tempo.

⚠️ Importante: Não embaralhar dados

In [ ]:
def split_temporal(df, data_corte='2017-01-01'):
    train = df[df['date'] < data_corte]
    test = df[df['date'] >= data_corte]

    return train, test

##5. Treinamento de Modelos

Testando diferentes algoritmos para encontrar o com melhor desempenho.

In [ ]:
def treinar_modelos(X_train, y_train, X_test, y_test):
    modelos = {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=50)
    }

    resultados = {}

    for nome, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)

        mae = mean_absolute_error(y_test, pred)
        rmse = np.sqrt(((y_test - pred) ** 2).mean())

        resultados[nome] = {
            "modelo": modelo,
            "RMSE": rmse
        }

        print(f"{nome} -> RMSE: {rmse:.2f}")

    return resultados

##6. Pipeline

- Feature engineering
- Split
- Treino
- Seleção do melhor modelo

In [ ]:
def pipeline_completo(df):
    df = criar_features(df)

    train, test = split_temporal(df)

    features = [
        'day','month','dayofweek','weekofyear',
        'lag_1','lag_7','rolling_mean_7'
    ]

    X_train = train[features]
    y_train = train['sales']

    X_test = test[features]
    y_test = test['sales']

    resultados = treinar_modelos(X_train, y_train, X_test, y_test)

    melhor = min(resultados.items(), key=lambda x: x[1]['RMSE'])

    return melhor[1]['modelo'], X_test, y_test

##7. Execução do Modelo e vizualização

- Executa o modelo
- Vizualiza os resultados

In [ ]:
def plot_previsao(y_test, pred):
    plt.figure(figsize=(12,5))
    plt.plot(y_test.values[:200], label='Real')
    plt.plot(pred[:200], label='Previsto')
    plt.legend()
    plt.title("Real vs Previsto")
    plt.show()

modelo, X_test, y_test = pipeline_completo(df)

pred = modelo.predict(X_test)

plot_previsao(y_test, pred)